In [ ]:
from google.colab import drive
drive.mount("<mount-point>")

In [ ]:
from sklearn.model_selection import KFold

import pandas as pd
import numpy as np
import datasets
import torch

import warnings
warnings.simplefilter(action='ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Preparing data

The current order is:
- 0: zero
- 1: sft
- 2: berzak

In [ ]:
rank_df_ord = pd.read_excel('<project-data-path>', index_col = 0)

# rank_df_ord = rank_df_ord.rename({
#     'article': 'passage',
#     'berzak_distractor': 'distractor_1',
#     'zero_distractor': 'distractor_2',
#     'sft_distractor': 'distractor_3',
#     'berzak': 'distractor_1_com',
#     'zero': 'distractor_2_com',
#     'sft': 'distractor_3_com'
# }, axis = 1)

rank_df_ord.head(1)

## Loading modules

❗ REMEMBER TO UNCOMMENT THE MODEL LOADING PART OF BARTSCORE.

### BLEU & ROUGE (F)

### BERTScore (F)

### BARTScore

### BLEURT

### NLI

In [ ]:
# !pip install -U "huggingface_hub[cli]"
# !hf download microsoft/deberta-large-mnli --local-dir <project-data-path> --cache-dir /tmp/cache

In [ ]:
device

In [ ]:
from transformers import pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipe = pipeline(
    model='microsoft/deberta-large-mnli',
    return_all_scores=True,
    device=device
    )

pipe({
    'text': 'this is a span, quite long one',
    'text_pair': 'okay, this is the distractor based on the given span'
})

## Compiling all metrics

In [ ]:
a_spans = rank_df_ord['berzak_span_full'].values
passages = rank_df_ord['passage'].values

berzak_b_com, zero_b_com, sft_com = zip(*rank_df_ord[['berzak_distractor_com', 'zero_distractor_com', 'sft_distractor_com']].values)
berzak_b, zero_b, sft_b = zip(*rank_df_ord[['berzak_distractor', 'zero_distractor', 'sft_distractor']].values)

demo_span = a_spans[1]
demo_b = berzak_b[1]

print(demo_span)
print(demo_b)

In [ ]:
from transformers import pipeline

def get_logical_inference(distractor, span):
    res = pipe({
        'text': distractor,
        'text_pair': span
    })
    contra_score, neutral_score, entail_score = [i['score'] for i in res]
    print(res)
    return {'entail': entail_score, 'neutral': neutral_score, 'contra': contra_score}

get_logical_inference(demo_b, demo_span)

In [ ]:
import json
import os
from collections import defaultdict
from tqdm.auto import tqdm
import torch
from concurrent.futures import ThreadPoolExecutor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. 配置参数
BATCH_SIZE = 8  # 可调整批大小
RESUME = True    # 是否从断点继续
OUTPUT_FILE = "<project-data-path>"
CHECKPOINT_FILE = "<project-data-path>"

# 2. 初始化数据结构
if RESUME and os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        all_metrics = defaultdict(dict, json.load(f))
    processed_indices = set(all_metrics.keys())
else:
    all_metrics = defaultdict(dict)
    processed_indices = set()

# 3. GPU加速的批量处理函数
def process_batch(batch_rows, device='cuda'):
    batch_results = {}

    with ThreadPoolExecutor() as executor:
        futures = []
        for _, row in batch_rows:
            if str(row['index']) in processed_indices:
                continue

            futures.append(executor.submit(
                process_single_row,
                row,
                device=device
            ))

        for future in tqdm(futures, desc="Processing batch"):
            idx_key, result = future.result()
            batch_results[idx_key] = result

    return batch_results

def process_single_row(row, device='cuda'):
    idx_key = row['index']
    passage = row['passage']

    spans = row[['berzak_span_full', 'zero_span_full', 'sft_span_full']]

    # distractors = row[['berzak_distractor', 'zero_distractor', 'sft_distractor']]
    distractors = row[['berzak_distractor_com', 'zero_distractor_com', 'sft_distractor_com']]

    # 使用GPU加速的metric计算（假设get_metric_dict支持GPU）
    with torch.cuda.device(device):
        metrics = {
            'berzak': {'pair': (spans[0], distractors[0]), 'metrics': get_logical_inference(
                distractors[0], spans[0]
                )},
            'zero': {'pair': (spans[1], distractors[1]), 'metrics': get_logical_inference(
                distractors[1], spans[1]
                )},
            'sft': {'pair': (spans[2], distractors[2]), 'metrics': get_logical_inference(
                distractors[2], spans[2]
                )}
        }

    return idx_key, metrics

# 4. 分批处理主循环
try:
    batch = []
    for i, row in tqdm(rank_df_ord.iterrows(), total=len(rank_df_ord)):
        if str(row['index']) in processed_indices:
            continue

        batch.append((i, row))

        if len(batch) >= BATCH_SIZE:
            batch_results = process_batch(batch)
            all_metrics.update(batch_results)

            # 实时保存结果
            with open(OUTPUT_FILE, 'w') as f:
                json.dump(all_metrics, f)

            # 保存检查点
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump({'last_index': i}, f)

            batch = []

    # 处理剩余数据
    if batch:
        batch_results = process_batch(batch)
        all_metrics.update(batch_results)
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_metrics, f)

except Exception as e:
    print(f"Error occurred: {str(e)}")
    print("Saving progress before exiting...")
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(all_metrics, f)
    raise

# 5. 最终输出
print(f"Processing completed. Results saved to {OUTPUT_FILE}")